# Creating an additional schema

Inside script to run need to create two scripts -> one SQL for creating table and one .sh for running SQL code

In [ ]:
%%writefile script_to_run/parking_usage_analytics.sql

-- This script should be run after clickhouse-init.sql as it assumes parking_db and parkman_user exist.

CREATE TABLE IF NOT EXISTS parking_db.parking_usage (
    timestamp DateTime,         -- Time when entry "sampled" (from RedisDB)
    owner_id String,            -- ID of parking lot owner (originates from MongoDB)
    owner_full_name String,     -- Name + surname of owner ( originates from MongoDB)
    parking_lot_id String,      -- ID of parking lot ( originates from TimescaleDB/MongoDB)
    parking_lot_name String,    -- Name of parking lot ( originates from MongoDB)
    parking_spot_number Int32,  -- Number of parking spots for the parking lot (from RedisDB, originates from MongoDB)
    car_count Int32             -- Number of counted cars on parking (from RedisDB)
)
ENGINE = MergeTree()
PARTITION BY (owner_id, toYYYYMM(timestamp)) -- Composite partition by owner_id and year-month of timestamp
ORDER BY (owner_id, parking_lot_id, timestamp); -- Order by owner, parking lot, and then by time

GRANT ALTER, SELECT, INSERT ON parking_db.parking_usage TO parkman_user;

In [11]:
%%writefile script_to_run/execute-parking-usage-init.sh
#!/bin/sh
set -e # Exit immediately if a command exits with a non-zero status.

SQL_SCRIPT_PATH="/tmp/clickhouse-init-parking-usage.sql" # Path inside the admin container
CLICKHOUSE_HOST="clickhouse" # Service name of your ClickHouse server
CLICKHOUSE_USER="default"    # User to connect as (default usually has admin rights)
CLICKHOUSE_PASSWORD=""       # Password for the user (leave empty if default user has no password)
                             # Adjust per users.xml (configures a password for the 'default' user).

ATTEMPTS=0
MAX_ATTEMPTS=12 # Try for 1 minute (12 attempts * 5 seconds = 60 seconds)

echo "Admin Container: Waiting for ClickHouse server ($CLICKHOUSE_HOST) to be ready..."

# Loop until ClickHouse is responsive or max attempts are reached
until [ "$ATTEMPTS" -ge "$MAX_ATTEMPTS" ] || clickhouse-client -h "$CLICKHOUSE_HOST" --user "$CLICKHOUSE_USER" --password "$CLICKHOUSE_PASSWORD" --query 'SELECT 1' >/dev/null 2>&1; do
  ATTEMPTS=$((ATTEMPTS+1))
  echo "Admin Container: ClickHouse not ready (attempt $ATTEMPTS/$MAX_ATTEMPTS), retrying in 5s..."
  sleep 5
done

if [ "$ATTEMPTS" -ge "$MAX_ATTEMPTS" ]; then
  echo "Admin Container: Failed to connect to ClickHouse server after $MAX_ATTEMPTS attempts. Exiting."
  exit 1
fi

echo "Admin Container: ClickHouse server is ready. Executing SQL script: $SQL_SCRIPT_PATH"
clickhouse-client -h "$CLICKHOUSE_HOST" --user "$CLICKHOUSE_USER" --password "$CLICKHOUSE_PASSWORD" --multiquery < "$SQL_SCRIPT_PATH"

echo "Admin Container: SQL script executed successfully. Container will now exit."
exit 0

Overwriting script_to_run/execute-parking-usage-init.sh


## Kreiranje docker-compose containera

In [10]:
%%writefile admin.clickhouse.docker-compose.yml
version: '3.8'

networks:
      app-network:
          external: true
services:
  clickhouse-init-parking-usage-admin:
    image: clickhouse/clickhouse-client:latest # Consider pinning to a specific version
    container_name: clickhouse-init-parking-usage-admin
    volumes:
      # Mount the SQL script to be executed (read-only)
      - ./script_to_run/parking_usage_analytics.sql:/tmp/clickhouse-init-parking-usage.sql:ro
      # Mount the helper script (read-only)
      - ./script_to_run/execute-parking-usage-init.sh:/usr/local/bin/execute-parking-usage-init.sh:ro
    entrypoint: /usr/local/bin/execute-parking-usage-init.sh
    networks:
      - app-network



Overwriting admin.clickhouse.docker-compose.yml


## Execute docker container

In [12]:
!docker compose -f admin.clickhouse.docker-compose.yml run --rm clickhouse-init-parking-usage-admin

WARN[0000] /home/benjamin/Documents/ParkMan/ClickHouseDB/AdminContainer/admin.clickhouse.docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
Admin Container: Waiting for ClickHouse server (clickhouse) to be ready...
Admin Container: ClickHouse server is ready. Executing SQL script: /tmp/clickhouse-init-parking-usage.sql
Admin Container: SQL script executed successfully. Container will now exit.
